# 03 — Fine-tuning DistilBERT (Google Colab GPU)

**Projet** : FakeNews Analyzer — DevComplex  
**⚠️ À exécuter sur Google Colab avec GPU T4 (Runtime → Modifier le type de runtime → T4 GPU)**

**Prérequis** : Uploader `09_data/colab_exports/` sur Google Drive avant d'exécuter.

---

In [6]:
from google.colab import files
uploaded = files.upload()  # sélectionner train_texts.csv et test_texts.csv

Saving train_texts.csv to train_texts.csv
Saving test_texts.csv to test_texts.csv


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# ============================================================
# Fichier  : 03_distilbert_finetune.ipynb
# Rôle     : Fine-tuning DistilBERT pour la classification FAKE/REAL
# Version  : V1 (Colab GPU)
# Projet   : FakeNews Analyzer — DevComplex
# Auteur   : DevComplex
# ============================================================
# Exécuter sur Google Colab avec GPU T4
# ============================================================

import os
import json
import time
import pandas as pd
import numpy as np

# Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/fakenews_colab'
OUTPUT_DIR = '/content/distilbert_output'

# CORRECTION : On crée les DEUX dossiers pour être tranquille
os.makedirs(DRIVE_DIR, exist_ok=True)  # Crée le dossier sur le Drive si manquant
os.makedirs(OUTPUT_DIR, exist_ok=True) # Crée le dossier temporaire local

print('✓ Google Drive monté et dossiers créés !')
print(f'Répertoire données (Drive) : {DRIVE_DIR}')
print(f'Répertoire output (Local) : {OUTPUT_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive monté et dossiers créés !
Répertoire données (Drive) : /content/drive/MyDrive/fakenews_colab
Répertoire output (Local) : /content/distilbert_output


In [9]:
# Installer les dépendances supplémentaires si nécessaire
# !pip install transformers datasets accelerate -q

import torch
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device : cuda
GPU : Tesla T4
VRAM : 15.6 GB


In [10]:
import os
import shutil

# 1. Définir et créer le dossier sur le Google Drive
DRIVE_DIR = '/content/drive/MyDrive/fakenews_colab'
os.makedirs(DRIVE_DIR, exist_ok=True)

# 2. Déplacer automatiquement les fichiers du stockage local vers le Drive
for filename in ['train_texts.csv', 'test_texts.csv']:
    chemin_local = os.path.join('/content', filename)
    chemin_drive = os.path.join(DRIVE_DIR, filename)

    if os.path.exists(chemin_local):
        shutil.move(chemin_local, chemin_drive)
        print(f"✓ {filename} déplacé avec succès vers le Drive.")
    elif os.path.exists(chemin_drive):
        print(f"✓ {filename} est déjà présent sur le Drive.")
    else:
        print(f"⚠️ Erreur : {filename} introuvable. Relancez la cellule files.upload()")

# 3. Maintenant, votre code de lecture fonctionnera sans erreur
import pandas as pd
train_df = pd.read_csv(os.path.join(DRIVE_DIR, 'train_texts.csv'))
test_df  = pd.read_csv(os.path.join(DRIVE_DIR, 'test_texts.csv'))
print("✓ Fichiers chargés avec succès dans les DataFrames !")


✓ train_texts.csv déplacé avec succès vers le Drive.
✓ test_texts.csv déplacé avec succès vers le Drive.
✓ Fichiers chargés avec succès dans les DataFrames !


## Section 1 — Chargement des données

In [11]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 256

train_df = pd.read_csv(os.path.join(DRIVE_DIR, 'train_texts.csv'))
test_df  = pd.read_csv(os.path.join(DRIVE_DIR, 'test_texts.csv'))

train_df = train_df.dropna(subset=['clean_text'])
test_df  = test_df.dropna(subset=['clean_text'])

print(f'Train : {len(train_df):,} | Test : {len(test_df):,}')
print(f'Distribution train — FAKE: {train_df["label"].sum():,} | REAL: {(train_df["label"]==0).sum():,}')

Train : 49,935 | Test : 12,052
Distribution train — FAKE: 22,273 | REAL: 27,662


## Section 2 — Tokenisation

In [12]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['clean_text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length'
    )

train_dataset = Dataset.from_pandas(train_df[['clean_text', 'label']])
test_dataset  = Dataset.from_pandas(test_df[['clean_text', 'label']])

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column('label', 'labels')
test_dataset  = test_dataset.rename_column('label', 'labels')
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f'✓ Tokenisation terminée')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/49935 [00:00<?, ? examples/s]

Map:   0%|          | 0/12052 [00:00<?, ? examples/s]

✓ Tokenisation terminée


## Section 3 — Entraînement

In [14]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'REAL', 1: 'FAKE'},
    label2id={'REAL': 0, 'FAKE': 1}
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probas = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='weighted'),
        'auc': roc_auc_score(labels, probas),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,                          # Mixed precision (GPU uniquement)
    eval_strategy='epoch',             # <-- CORRECTION ICI (au lieu de evaluation_strategy)
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=100,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Démarrage de l\'entraînement...')
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Démarrage de l'entraînement...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Auc
1,0.127615,0.106337,0.957683,0.957612,0.993746
2,0.064851,0.148822,0.955692,0.955518,0.994729
3,0.044506,0.126421,0.965649,0.965627,0.995242


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=4683, training_loss=0.10396110230013521, metrics={'train_runtime': 1040.9498, 'train_samples_per_second': 143.912, 'train_steps_per_second': 4.499, 'total_flos': 9922139327831040.0, 'train_loss': 0.10396110230013521, 'epoch': 3.0})

## Section 4 — Évaluation finale et sauvegarde

In [15]:
results = trainer.evaluate()
print('\n=== RÉSULTATS FINAUX DistilBERT ===')
for k, v in results.items():
    print(f'  {k} : {v:.4f}' if isinstance(v, float) else f'  {k} : {v}')


=== RÉSULTATS FINAUX DistilBERT ===
  eval_loss : 0.1264
  eval_accuracy : 0.9656
  eval_f1 : 0.9656
  eval_auc : 0.9952
  eval_runtime : 22.0344
  eval_samples_per_second : 546.9630
  eval_steps_per_second : 8.5770
  epoch : 3.0000


In [17]:
import os
import shutil
from google.colab import files

# 1. Sauvegarder le modèle et le tokenizer en local
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✓ Modèle sauvegardé dans {OUTPUT_DIR}')

# 2. Créer l'archive ZIP en local
zip_path = '/content/distilbert_model.zip'
shutil.make_archive('/content/distilbert_model', 'zip', OUTPUT_DIR)
print(f'✓ Archive créée : {zip_path}')

# 3. SOLUTION SÉCURISÉE : Copier l'archive directement sur votre Drive
# (Remplacez DRIVE_DIR si le nom de votre variable de dossier Drive est différent)
DRIVE_DIR = '/content/drive/MyDrive/fakenews_colab'
os.makedirs(DRIVE_DIR, exist_ok=True)

chemin_destination_drive = os.path.join(DRIVE_DIR, 'distilbert_model.zip')
shutil.copy(zip_path, chemin_destination_drive)
print(f'✓ Archive sauvegardée de façon permanente sur votre Drive : {chemin_destination_drive}')

# 4. Optionnel : Lancement du téléchargement classique par le navigateur
try:
    print('Tentative de téléchargement direct via le navigateur...')
    files.download(zip_path)
    print('✓ Téléchargement lancé — dézipper dans 04_models/distilbert/')
except Exception as e:
    print("⚠️ Le téléchargement direct a échoué (normal pour les gros fichiers), mais votre fichier est en sécurité sur votre Google Drive !")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Modèle sauvegardé dans /content/distilbert_output
✓ Archive créée : /content/distilbert_model.zip
✓ Archive sauvegardée de façon permanente sur votre Drive : /content/drive/MyDrive/fakenews_colab/distilbert_model.zip
Tentative de téléchargement direct via le navigateur...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Téléchargement lancé — dézipper dans 04_models/distilbert/
